# Ingesta de Datos del Registro Público de Derechos de Agua (REPDA)

Esta libreta de Jupyter se utilizará para la ingesta y análisis de datos relacionados con las concesiones de agua superficial y subterránea registradas ante la CONAGUA a través del REPDA. Para ello, utilizaremos los datos disponibles para conocer el volumen concesionado para uso agrícola en el estado de Sonora.

## Ingesta de datos

### Librerías necesarias

In [1]:
#!pip install -q pandas requests urllib3 pyarrow pydantic

In [2]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from pydantic import BaseModel, PositiveInt, ValidationError, Field, ConfigDict, field_validator
import pandas as pd
import requests
import urllib3
from typing import Optional
from loguru import logger
import time

# Disable SSL certificate warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

root = Path().resolve().parent  
raw_dir = root / "data/raw/"
interim_dir = root / "data/interim/"
interim_dir.mkdir(parents=True, exist_ok=True)

complete_csv_output = raw_dir / "repda_sonora_completo.csv"
agricultural_csv_output = interim_dir / "repda_sonora_agricola.csv"
output_parquet = raw_dir / "repda_sonora_completo.parquet"

print("¡Librerías importadas y directorios listos!")

¡Librerías importadas y directorios listos!


### Descarga de datos

In [3]:
url_query = (
    "https://sigagis.conagua.gob.mx/ArcGIS/rest/services/REPDA/MapServer/1/query"
)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Referer": "https://sigagis.conagua.gob.mx/",
}

where_clause = "ESTADO = 26"
batch_size = 2000

# Check the total number of exact records first
print("Consultando el número total de registros en Sonora...")
count_params = {"where": where_clause, "returnCountOnly": "true", "f": "json"}
r = requests.get(
    url_query, params=count_params, headers=headers, verify=False, timeout=15
)
total_records = r.json().get("count", 0)
print(f"Total de registros a descargar: {total_records}")


# Function to download a batch in parallel
def fetch_chunk(offset):
    params = {
        "where": where_clause,
        "outFields": "*",
        "f": "json",
        "returnGeometry": "false",
        "resultRecordCount": batch_size,
        "resultOffset": offset,
        "orderByFields": "FID",
    }
    for _ in range(3):
        try:
            resp = requests.get(
                url_query, params=params, headers=headers, verify=False, timeout=20
            )
            if resp.status_code == 200:
                data = resp.json()
                return [feat["attributes"] for feat in data.get("features", [])]
        except Exception:
            time.sleep(1)
    return []


# Run bulk download
offsets = list(range(0, total_records, batch_size))
all_features = []

logger.info("Descargando todo Sonora en paralelo...")
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(fetch_chunk, off): off for off in offsets}
    for future in as_completed(futures):
        res = future.result()
        if res:
            all_features.extend(res)
        logger.success(
            f"Registros descargados: {len(all_features)} / {total_records}", end="\r"
        )

logger.success(
    "\n¡Descarga finalizada con éxito! Total de registros obtenidos:"
    f" {len(all_features)}"
)


Consultando el número total de registros en Sonora...


2026-09-28 20:44:04.523 | INFO     | __main__:<module>:51 - Descargando todo Sonora en paralelo...


Total de registros a descargar: 19274


2026-09-28 20:44:06.173 | SUCCESS  | __main__:<module>:58 - Registros descargados: 1000 / 19274
2026-09-28 20:44:06.235 | SUCCESS  | __main__:<module>:58 - Registros descargados: 2000 / 19274
2026-09-28 20:44:07.506 | SUCCESS  | __main__:<module>:58 - Registros descargados: 3000 / 19274
2026-09-28 20:44:07.562 | SUCCESS  | __main__:<module>:58 - Registros descargados: 4000 / 19274
2026-09-28 20:44:08.713 | SUCCESS  | __main__:<module>:58 - Registros descargados: 5000 / 19274
2026-09-28 20:44:08.772 | SUCCESS  | __main__:<module>:58 - Registros descargados: 6000 / 19274
2026-09-28 20:44:10.327 | SUCCESS  | __main__:<module>:58 - Registros descargados: 7000 / 19274
2026-09-28 20:44:10.355 | SUCCESS  | __main__:<module>:58 - Registros descargados: 8000 / 19274
2026-09-28 20:44:12.244 | SUCCESS  | __main__:<module>:58 - Registros descargados: 9000 / 19274
2026-09-28 20:44:12.245 | SUCCESS  | __main__:<module>:58 - Registros descargados: 10000 / 19274
2026-09-28 20:44:12.246 | SUCCESS  | __

### Procesamiento, Limpieza y Filtrado Agrícola

In [4]:
if all_features:
  df_sonora = pd.DataFrame(all_features)

  if "FID" in df_sonora.columns:
    df_sonora = df_sonora.drop_duplicates(subset=["FID"])

  # Save full data backup
  df_sonora.to_csv(complete_csv_output, index=False, encoding="utf-8-sig")
  df_sonora.to_parquet(output_parquet, index=False)
  logger.info(f"[1/2] Archivo completo guardado en: {complete_csv_output.name}")

  # Detect water usage column and normalize text
  col_uso = None
  for c in ["USO_SUB", "USO", "USO_EXTRACCION"]:
    if c in df_sonora.columns:
      col_uso = c
      break

  if col_uso:
    df_sonora["USO_LIMPIO"] = (
        df_sonora[col_uso]
        .astype(str)
        .str.upper()
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
        .str.strip()
    )
    df_agricola = df_sonora[
        df_sonora["USO_LIMPIO"].str.contains("AGRICOLA", na=False)
    ].copy()
  else:
    df_agricola = df_sonora.copy()

  # Save filtered file for agriculture
  df_agricola.to_csv(agricultural_csv_output, index=False, encoding="utf-8-sig")
  logger.info(f"[2/2] Archivo agrícola guardado en: {agricultural_csv_output.name}")
  logger.info(f"Total de títulos con uso agrícola: {len(df_agricola)}")

  # Show a quick preview in the notebook
  display(df_agricola.head(3))
else:
  logger.warning("No se obtuvieron registros para procesar.")

2026-09-28 20:44:12.313 | INFO     | __main__:<module>:10 - [1/2] Archivo completo guardado en: repda_sonora_completo.csv
2026-09-28 20:44:12.320 | INFO     | __main__:<module>:37 - [2/2] Archivo agrícola guardado en: repda_sonora_agricola.csv
2026-09-28 20:44:12.321 | INFO     | __main__:<module>:38 - Total de títulos con uso agrícola: 94


,FID,LONDEC,LATDEC,REGIONAL,FOLIOREG,TITULAR,ESTADO,MUPIO,USO_SUB,CVE_ACUHOM,ACU_SUBHOM,VOL_CONS,NUM,USO_LIMPIO
0,214547,-114.943333,32.366667,1,A1BCA100625/07AMGE94,"USUARIOS DEL POZO PARTICULAR # 628, CLAVE (28-C)",26,55,AGRICOLA,210,VALLE DE MEXICALI,676834.0,1,AGRICOLA
1,214548,-114.930000,32.345833,1,A1BCA100691/07AMGE94,"USUARIOS DEL POZO PARTICULAR # 794, CLAV",26,55,AGRICOLA,210,VALLE DE MEXICALI,496345.0,1,AGRICOLA
2,214549,-114.788889,32.477778,1,A1BCA100807/07AMGE94,"USUARIOS DEL POZO PARTICULAR # 827, CLAV",26,55,AGRICOLA,210,VALLE DE MEXICALI,58659.0,1,AGRICOLA


## Validación de la descarga de datos

### Schema basandonos en el diccionario de datos 

In [5]:
class RepdaAgricolaSchema(BaseModel):
  model_config = ConfigDict(extra="ignore")

  FID: Optional[int] = Field(default=None, ge=0)
  NUM_TITULO: Optional[str] = None
  NUM_APROVE: Optional[str] = None
  NOMBRE: Optional[str] = None
  ESTADO: Optional[int] = Field(default=None, ge=1)
  CLAVE_MUN: Optional[int] = Field(default=None, ge=1)
  MUNICIPIO: Optional[str] = None
  LOCALIDAD: Optional[str] = None
  ACUIFERO: Optional[str] = None
  CUENCA: Optional[str] = None
  USO: Optional[str] = None
  USO_SUB: Optional[str] = None
  USO_LIMPIO: Optional[str] = None
  VOL_CONS: Optional[float] = Field(default=None, ge=0)
  FECHA_HASTA: Optional[str] = None  # Can be adjusted to datetime if preferred
  LATITUD: Optional[float] = None
  LONGITUDE: Optional[float] = None

  @field_validator("*")
  @classmethod
  def normalize_missing_values(cls, value):
    if value is None or pd.isna(value):
      return None
    return value

  @field_validator(
      "NUM_TITULO",
      "NUM_APROVE",
      "NOMBRE",
      "MUNICIPIO",
      "LOCALIDAD",
      "ACUIFERO",
      "CUENCA",
      "USO",
      "USO_SUB",
      "USO_LIMPIO",
      "FECHA_HASTA",
      mode="before",
  )
  @classmethod
  def convert_text_fields(cls, value):
    if value is None or pd.isna(value):
      return None
    return str(value).strip()

### Función para validar los datos descargados

In [6]:
def validate_df(df: pd.DataFrame) -> pd.DataFrame:
    validated_rows = []

    for index, row in df.iterrows():
        try:
            agriculture = RepdaAgricolaSchema(**row.to_dict())
            validated_rows.append(agriculture.model_dump())

        except ValidationError as e:
            print(f"Error en fila {index}:")
            print(e)

    return pd.DataFrame(validated_rows)

In [7]:
df_validated = validate_df(df_agricola)
df_validated.head(5)

,FID,NUM_TITULO,NUM_APROVE,NOMBRE,ESTADO,CLAVE_MUN,MUNICIPIO,LOCALIDAD,ACUIFERO,CUENCA,USO,USO_SUB,USO_LIMPIO,VOL_CONS,FECHA_HASTA,LATITUD,LONGITUDE
0,214547,None,None,None,26,None,None,None,None,None,None,AGRICOLA,AGRICOLA,676834.0,None,None,None
1,214548,None,None,None,26,None,None,None,None,None,None,AGRICOLA,AGRICOLA,496345.0,None,None,None
2,214549,None,None,None,26,None,None,None,None,None,None,AGRICOLA,AGRICOLA,58659.0,None,None,None
3,214550,None,None,None,26,None,None,None,None,None,None,AGRICOLA,AGRICOLA,861836.0,None,None,None
4,214551,None,None,None,26,None,None,None,None,None,None,AGRICOLA,AGRICOLA,906958.0,None,None,None
